# Exploratory Data Analysis 

**How to use this notebook**
1. Set `RAW_DATA_PATH` and `DATASETS_DIR` variables in .env file.
2. Run cells top to bottom.
3. Each section ends with a markdown cell where you **write down findings and decisions** — that running log is the real deliverable of an EDA, not the plots.

> Order of work: understand the problem first, assess data *quality* before you trust any distribution, and let the domain drive interpretation.



**Context & questions (fill this in before touching the data)**

- **Goal / downstream task**: Describe and Clean discharge_notes_with_meds dataset which contains Admission and Discharge Medication presented in MIMIC-IV Discharge Notes
- **Unit of observation** (one row = ?): 1 Discharge Clinical Note

## 1. Imports & Setup

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from clinical_notes_extraction.utils.eda import classify_age



In [ ]:
load_dotenv()

## 2. Load Raw Data

In [ ]:
df = pd.read_parquet(f'{os.environ["DATA_PATH"]}/raw/discharge_notes_with_meds.parquet')
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")


## 3. Structural Overview

**Check:**
- size 
- types 
- memory

**Look at:** 
- head
- tail
- data types

In [ ]:
print("Shape:", df.shape)
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB\n")


In [ ]:
df.info()

In [ ]:
display(df.head())

In [ ]:
display(df.tail())

In [ ]:
display(df.sample(min(5, len(df)), random_state=0))

In [ ]:
# dtype breakdown
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
dt_cols  = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
print(f"Numeric ({len(num_cols)}): {num_cols}")
print(f"Categorical ({len(cat_cols)}): {cat_cols}")
print(f"Datetime ({len(dt_cols)}): {dt_cols}")

In [ ]:
df1 = df.copy()

### 3.1 Update data types

Update data types from numeric to categorical in subject_id and hadm_id columns.

In [ ]:
data_types_cols = ['subject_id', 'hadm_id']

df1[data_types_cols] = df1[data_types_cols].astype("category")
        

In [ ]:
df1.info()

In [ ]:
# dtype breakdown
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
dt_cols  = df.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist()
print(f"Numeric ({len(num_cols)}): {num_cols}")
print(f"Categorical ({len(cat_cols)}): {cat_cols}")
print(f"Datetime ({len(dt_cols)}): {dt_cols}")

### 3.2 Add new columns

- meds_on_admission_length
- meds_on_discharge_length
- patient_age_group

#### 3.2.1 Add columns:
- meds_on_admission_length
- meds_on_discharge_length

In [ ]:
medication_cols = ['meds_on_admission', 'meds_on_discharge']


for col in medication_cols:
    new_column = f'{col}_length'

    df1[new_column] = df[col].str.len()

df1.dtypes

In [ ]:
df1.info()

#### 3.2.2 Add columns:
- patient_age_group_on_admission
- patient_age_group_on_discharge

**New column patient_age_group_on_admission:**

In [ ]:
df1['patient_age_on_admission'].describe()

In [ ]:
# Creation of an histogram to see the distribution of Patient Age on Admission:

plt.hist(df1['patient_age_on_admission'], bins=30, edgecolor='black')
plt.xlabel('Patient Age on Admission')
plt.ylabel('Frequency')
plt.title('Distribution of Patient Age on Admission')
plt.show()

**Note:**

As we can see in summary statistics table and in the histogram: 
- the patient age average is 61,67 years, which means ~62 years.
- the patient age median is 63 years.

In [ ]:
# Create age group column named patient_age_group_on_admission

# 1. Apply the function to 'patient_age_on_admission' column to create the 'patient_age_group_on_admission'
df1['patient_age_group_on_admission'] = df1['patient_age_on_admission'].apply(classify_age)

# 2. Verify the result
print(df1[['patient_age_on_admission', 'patient_age_group_on_admission']].head())

----------

**New column patient_age_group_on_discharge:**

In [ ]:
df1['patient_age_on_discharge'].describe()

In [ ]:
# Creation of an histogram to see the distribution of Patient Age on Admission:

plt.hist(df1['patient_age_on_admission'], bins=30, edgecolor='black')
plt.xlabel('Patient Age on Admission')
plt.ylabel('Frequency')
plt.title('Distribution of Patient Age on Admission')
plt.show()

**Note:**

As we can see in summary statistics table and in the histogram: 
- the patient age average is 61,69 years, which means ~62 years.
- the patient age median is 63 years.

In [ ]:
# Create age group column named patient_age_group_on_discharge

# 1. Apply the function to 'patient_age_on_admission' column to create the 'patient_age_group_on_admission'
df1['patient_age_group_on_discharge'] = df1['patient_age_on_discharge'].apply(classify_age)

# 2. Verify the result
print(df1[['patient_age_on_discharge', 'patient_age_group_on_discharge']].head())

In [ ]:
df1.tail()

## 4. Duplicates & Key integrity

In [ ]:
dups = df1.duplicated().sum()
print(f"Fully duplicated rows: {dups:,} ({dups/len(df):.1%})")

# If you have a key/id column, check it is unique:
KEY = "note_id"   # e.g. "patient_id"

if KEY:
    n_dup_keys = df[KEY].duplicated().sum()
    print(f"Duplicate {KEY} values: {n_dup_keys:,}")


## 5. Create 2 datasets -> Admission Medication Dataset & Discharge Medication Dataset

- One dataset should contains the relevant columns for admission medication and the other one should contains the useful columns for discharge medication.

In [ ]:
df1.info()

In [ ]:
df1.columns

In [ ]:
final_path = f'{os.environ["DATA_PATH"]}/datasets'

# Creates the entire folder structure; does nothing if they already exist
os.makedirs(final_path, exist_ok=True)

**Creation of medication_on_admission dataset:**

In [ ]:
final_admission_cols = [
    'note_id', 
    'subject_id', 
    'hadm_id', 
    'unit_type', 
    'patient_gender', 
    'patient_age_on_admission', 
    'patient_age_group_on_admission', 
    'meds_on_admission', 
    'meds_on_admission_length',
    'text' 
]

df1[final_admission_cols].to_parquet(f'{final_path}/medication_on_admission.parquet')

**Creation of medication_on_discharge dataset:**

In [ ]:
final_discharge_cols = [
    'note_id', 
    'subject_id', 
    'hadm_id', 
    'unit_type', 
    'patient_gender', 
    'patient_age_on_discharge', 
    'patient_age_group_on_discharge', 
    'meds_on_discharge', 
    'meds_on_discharge_length',
    'text' 
]

df1[final_discharge_cols].to_parquet(f'{final_path}/medication_on_discharge.parquet')